# Data Preparation and Structural Cleaning: Tasks 1 to 11

**Prepared by:** Ana Abreu Estevez (Tasks 1 to 11: data preparation and structural cleaning)

**In collaboration with:** Trinity Thao (Tasks 12 to 22: content cleaning and final validation)

**Project:** NYC Restaurant Inspections (DSBA/MBAD 6211)

This notebook documents the data preparation and structural cleaning of the raw inspection file. Every step is recorded so that teammates and reviewers can reproduce the results from the original CSV.

## Task 1: Preserve the Original Dataset and Define the Cleaning Rules

| Item | Detail |
|---|---|
| Source | NYC Department of Health and Mental Hygiene (DOHMH) Restaurant Inspection Results, published on NYC Open Data |
| Original file name | DOHMH_New_York_City_Restaurant_Inspection_Results_20260827.csv |
| File name in this project | data/restaurant_inspections.csv |
| Snapshot date | 2026-08-27 |

**Cleaning rules**

| Situation | What we do |
|---|---|
| Original CSV | Never changed. We clean a copy. |
| Exact duplicate rows | Can be removed from the copy, and each one is logged with the reason. |
| Several violations in one inspection | Kept. Each violation is its own row. |
| Placeholder rows (date 01/01/1900) | Flagged, not erased. They mean "not inspected yet". |
| Missing scores, grades, cuisine, violation details | Kept for Trinity's review. Not filled in or removed. |
| Columns with no useful information | Dropped from the copy only if the team agrees, and documented. |
| Every decision | Written down so others can repeat it. |

In [27]:
import pandas as pd

df = pd.read_csv("data/DOHMH_New_York_City_Restaurant_Inspection_Results_20260827.csv", low_memory=False)

print(df.shape)

(296012, 27)


296,012 = rows. Each row is one violation found during an inspection, not one restaurant. If an inspection found 4 violations, that inspection takes 4 rows.
27 = columns. These are the details recorded for each row (restaurant name, borough, inspection date, score, grade, violation code, and so on).

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296012 entries, 0 to 296011
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   CAMIS                  296012 non-null  int64  
 1   DBA                    296010 non-null  object 
 2   BORO                   296012 non-null  object 
 3   BUILDING               295029 non-null  object 
 4   STREET                 295977 non-null  object 
 5   ZIPCODE                292930 non-null  float64
 6   PHONE                  295914 non-null  object 
 7   CUISINE DESCRIPTION    292218 non-null  object 
 8   INSPECTION DATE        296012 non-null  object 
 9   ACTION                 292298 non-null  object 
 10  VIOLATION CODE         289568 non-null  object 
 11  VIOLATION DESCRIPTION  289568 non-null  object 
 12  CRITICAL FLAG          296012 non-null  object 
 13  SCORE                  278786 non-null  float64
 14  GRADE                  145340 non-nu

In [29]:
summary = pd.DataFrame({
    "data_type": df.dtypes,
    "non_null": df.notnull().sum(),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().sum() / len(df) * 100).round(2),
    "unique_values": df.nunique()
})

summary

,data_type,non_null,missing,missing_pct,unique_values
CAMIS,int64,296012,0,0.00,31366
DBA,object,296010,2,0.00,24628
BORO,object,296012,0,0.00,6
BUILDING,object,295029,983,0.33,8558
STREET,object,295977,35,0.01,2311
ZIPCODE,float64,292930,3082,1.04,256
PHONE,object,295914,98,0.03,27705
CUISINE DESCRIPTION,object,292218,3794,1.28,90
INSPECTION DATE,object,296012,0,0.00,1966
ACTION,object,292298,3714,1.25,5


## Step 2: Missing values

The file has 296,012 rows and 27 columns. Each row is one violation, not one inspection.

**Findings**
- GRADE (51% missing) and GRADE DATE (54% missing) are empty by design. A grade is only issued on some inspections, so we will not fill these in.
- VIOLATION CODE and DESCRIPTION are missing in 6,444 rows (2%). These rows have no violation to describe.
- ACTION and INSPECTION TYPE are each missing in 3,714 rows. This matches the 3,714 rows with the placeholder date 01/01/1900 (restaurants not yet inspected).
- SCORE is missing in 17,226 rows (6%).
- All other columns are missing less than 2% (cuisine, zipcode, census tract, neighborhood, latitude and longitude).

**Data type problems to fix**
- INSPECTION DATE, GRADE DATE and RECORD DATE are stored as text and need to become dates.
- ZIPCODE (and likely Census Tract) is stored as a number, so leading zeros can be lost. These should be text.

**Decision:** we will not drop every row with a missing value, because that would remove most of the data. We will decide column by column.

In [30]:
pd.set_option("display.max_columns", None)

df.sample(20, random_state=1)

,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,SCORE,GRADE,GRADE DATE,RECORD DATE,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
260149,50130607,OUR BACKYARD CAFE,Queens,46-10,104 STREET,11368.0,7185659104,Bakery Products/Desserts,11/07/2024,Violations were cited in the following area(s).,10B,Anti-siphonage or back-flow prevention device ...,Not Critical,27.0,B,11/07/2024,08/26/2026,Cycle Inspection / Re-inspection,40.746230,-73.859835,404.0,21.0,41300.0,4437329.0,4.019810e+09,QN25,POINT (-73.859834714739 40.746230036241)
201461,40387861,TROOP WINGS,Manhattan,614,9 AVENUE,10036.0,2125827661,American,05/29/2025,Violations were cited in the following area(s).,04N,Filth flies or food/refuse/sewage associated w...,Critical,25.0,NaN,NaN,08/26/2026,Cycle Inspection / Initial Inspection,40.759479,-73.991849,104.0,3.0,12100.0,1024979.0,1.010340e+09,MN15,POINT (-73.99184931572 40.759479232348)
252509,50173198,NEW MAY MAY KITCHEN,Manhattan,181,CLINTON STREET,10002.0,2124201328,Chinese,11/24/2025,Violations were cited in the following area(s).,08A,Establishment is not free of harborage or cond...,Not Critical,33.0,NaN,NaN,08/26/2026,Pre-permit (Operational) / Initial Inspection,40.715431,-73.987010,103.0,1.0,1401.0,1077606.0,1.003110e+09,MN28,POINT (-73.987010122947 40.715430818244)
110071,50181493,HALAL MUNCHIES,Manhattan,1810,3 AVENUE,10029.0,9298901198,Sandwiches,05/13/2026,Establishment Closed by DOHMH. Violations were...,04L,Evidence of mice or live mice in establishment...,Critical,18.0,NaN,NaN,08/26/2026,Pre-permit (Operational) / Reopening Inspection,40.788040,-73.947258,111.0,8.0,16600.0,1087540.0,1.016288e+09,MN33,POINT (-73.947257743984 40.788040307017)
87346,41427508,KING OF TANDOOR,Brooklyn,600,FLATBUSH AVENUE,11225.0,3475336811,Indian,06/08/2026,Violations were cited in the following area(s).,04L,Evidence of mice or live mice in establishment...,Critical,23.0,NaN,NaN,08/26/2026,Cycle Inspection / Initial Inspection,40.658951,-73.960541,309.0,40.0,79802.0,3114605.0,3.050260e+09,BK60,POINT (-73.960540875347 40.658951036849)
150453,41300703,PAO DE QUEIJO & CO.,Queens,31-90,30 STREET,11106.0,7182041979,Brazilian,07/31/2024,Violations were cited in the following area(s).,02G,Cold TCS food item held above 41 °F; smoked or...,Critical,18.0,NaN,NaN,08/26/2026,Cycle Inspection / Initial Inspection,40.762516,-73.926059,401.0,22.0,7500.0,4445865.0,4.005880e+09,QN70,POINT (-73.92605928813 40.762516107761)
51560,50145402,JR SUSHI,Manhattan,119,CHAMBERS STREET,10007.0,2122338338,Japanese,02/24/2025,Violations were cited in the following area(s).,02B,Hot TCS food item not held at or above 140 °F.,Critical,26.0,Z,02/24/2025,08/26/2026,Cycle Inspection / Re-inspection,40.715225,-74.008585,101.0,1.0,2100.0,1001581.0,1.001458e+09,MN24,POINT (-74.008585340645 40.715225374917)
115142,50166044,SANDZ,Queens,234-32,MERRICK BOULEVARD,11422.0,3479442511,Caribbean,12/01/2025,Violations were cited in the following area(s).,10G,Dishwashing and ware washing: Cleaning and san...,Not Critical,21.0,N,NaN,08/26/2026,Pre-permit (Operational) / Initial Inspection,40.674307,-73.735702,413.0,31.0,63200.0,4539017.0,4.131980e+09,QN66,POINT (-73.735701520645 40.674306554243)
203430,50045623,LUCKY STAR,Brooklyn,2149,KNAPP STREET,11229.0,7183325282,Chinese,04/23/2026,Violations were cited in the following area(s).,02G,Cold TCS food item held above 41 °F; smoked or...,Critical,11.0,A,04/23/2026,08/26/2026,Cycle Inspection / Initial Inspection,40.600242,-73.934083,315.0,46.0,57000.0,3248258.0,3.088220e+09,BK17,POINT (-73.934082777456 40.60024166899)
154216,50138275,MURPHY'S BAR,Queens,45-01,23 AVENUE,11105.0,3472423786,New American,03/25/2026,Violations were cited in the following area(s).,20-06,Current letter grade or Grade Pending card not...,Not Critical,NaN,NaN,NaN,08/26/2026,Administrative Miscellaneous / Initial Insp

In [31]:
print("All restaurants:", df["CAMIS"].nunique())

inspected = df[df["INSPECTION DATE"] != "01/01/1900"]
print("Restaurants with a real inspection:", inspected["CAMIS"].nunique())

print(inspected.groupby("BORO")["CAMIS"].nunique())

All restaurants: 31366
Restaurants with a real inspection: 27652
BORO
0                   32
Bronx             2425
Brooklyn          7077
Manhattan        10850
Queens            6268
Staten Island     1000
Name: CAMIS, dtype: int64


In [32]:
print("Restaurants with at least one score:", df[df["SCORE"].notna()]["CAMIS"].nunique())

inspections = df[df["INSPECTION DATE"] != "01/01/1900"].drop_duplicates(
    subset=["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"]
)

print("Inspections:", len(inspections))
print("Inspections with a score:", inspections["SCORE"].notna().sum())
print(inspections["SCORE"].describe())

Restaurants with at least one score: 27647
Inspections: 93292
Inspections with a score: 82956
count    82956.000000
mean        17.890388
std         14.922439
min          0.000000
25%          9.000000
50%         13.000000
75%         24.000000
max        214.000000
Name: SCORE, dtype: float64


### Create a working copy

**Intention:** keep the original table (`df`) exactly as it was loaded, and do all cleaning on a separate copy (`df_work`). If a cleaning step goes wrong, the original is still available to compare against or restart from.

In [33]:
import pandas as pd

df = pd.read_csv("data/DOHMH_New_York_City_Restaurant_Inspection_Results_20260827.csv", low_memory=False)
df_work = df.copy()

print("Original (df):", df.shape)
print("Working copy: ", df_work.shape)

Original (df): (296012, 27)
Working copy:  (296012, 27)


## Task 2: Record the Starting Baseline

**Intention:** record the key numbers of the raw data (rows, columns, unique restaurants, missing values and exact duplicates) before any cleaning. These are the "before" numbers used to check that cleaning did not remove records by accident.

In [34]:
baseline = {
    "rows": len(df),
    "columns": df.shape[1],
    "unique_restaurants_CAMIS": df["CAMIS"].nunique(),
    "empty_cells_total": int(df.isna().sum().sum()),
    "rows_with_at_least_one_empty_cell": int(df.isna().any(axis=1).sum()),
    "exact_duplicate_rows": int(df.duplicated().sum()),
}

pd.Series(baseline, name="raw_data")

rows                                 296012
columns                                  27
unique_restaurants_CAMIS              31366
empty_cells_total                    391745
rows_with_at_least_one_empty_cell    164755
exact_duplicate_rows                    169
Name: raw_data, dtype: int64

### Task 2 (continued): What the Raw Data Looks Like Before Cleaning

Before we clean anything, we want to understand what we are working with. In this part we look at three things that matter for our project: **cuisine**, **borough** and **score**. We use the raw data (`df`) on purpose, so these are our "before" numbers. After cleaning, we run the same code again and compare, to see what the cleaning changed.

#### Restaurants by borough

**What we want to know:** which boroughs have the most restaurants and which have the fewest, and what share of all restaurants each one holds.

**How we do it:** we count the restaurants in each borough (one row per restaurant) and sort them from the largest to the smallest. We also calculate the percentage of the total. A borough called `0` means the borough is unknown in the data.

In [35]:
restaurants = inspected.drop_duplicates(subset="CAMIS")
cuisine_counts = restaurants["CUISINE DESCRIPTION"].value_counts()

print("Restaurants in this table:", len(restaurants))
print("Different cuisines:", len(cuisine_counts))
cuisine_counts.head(10)

Restaurants in this table: 27652
Different cuisines: 90


CUISINE DESCRIPTION
American                    4853
Chinese                     2290
Coffee/Tea                  2220
Pizza                       1616
Mexican                     1111
Latin American              1096
Japanese                    1065
Italian                     1000
Bakery Products/Desserts     924
Caribbean                    747
Name: count, dtype: int64

In [36]:
borough_counts = restaurants["BORO"].value_counts()
borough_pct = (restaurants["BORO"].value_counts(normalize=True) * 100).round(1)

borough_table = pd.DataFrame({"restaurants": borough_counts, "percent": borough_pct})
borough_table

,restaurants,percent
BORO,,
Manhattan,10850,39.2
Brooklyn,7077,25.6
Queens,6268,22.7
Bronx,2425,8.8
Staten Island,1000,3.6
0,32,0.1


**What I found:**
1. Manhattan has the most restaurants (10,850, or 39.2% of the total), followed by Brooklyn (7,077, 25.6%) and Queens (6,268, 22.7%). Together these three boroughs hold about 87.5% of all restaurants with a real inspection.
2. The Bronx has 2,425 (8.8%) and Staten Island has the fewest with 1,000 (3.6%).
3. There are also 32 restaurants (0.1%) with an unknown borough (`0`). I review them in Task 10.

**Why it matters:** the boroughs are very different in size, so raw counts are not a fair way to compare them. In my reading, a borough with more restaurants will also have more violations just because of its size, so later I should compare averages or rates, such as the average score per borough. Staten Island has only 1,000 restaurants, so its results may be less stable than Manhattan's (this is my reasoning, not something I tested).

#### Cuisine by borough

**What we want to know:** how the 10 most common cuisines are spread across the five boroughs. For example, how many Japanese restaurants are in each borough?

**How we do it:** we build a table where each row is a cuisine and each column is a borough. Each number is how many restaurants have that cuisine in that borough. The **Total** column adds each row, and it should match the cuisine counts we calculated before. `Unknown` means the borough is `0` in the data.

In [37]:
top10 = cuisine_counts.head(10).index

cuisine_by_borough = pd.crosstab(restaurants["CUISINE DESCRIPTION"], restaurants["BORO"]).loc[top10]
cuisine_by_borough = cuisine_by_borough.rename(columns={"0": "Unknown"})
cuisine_by_borough = cuisine_by_borough[["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "Unknown"]]
cuisine_by_borough["Total"] = cuisine_by_borough.sum(axis=1)
cuisine_by_borough.index.name = "Cuisine"
cuisine_by_borough.columns.name = None

cuisine_by_borough

,Manhattan,Brooklyn,Queens,Bronx,Staten Island,Unknown,Total
Cuisine,,,,,,,
American,2470,1068,815,326,166,8,4853
Chinese,546,653,765,232,94,0,2290
Coffee/Tea,1113,608,348,88,59,4,2220
Pizza,484,423,351,253,104,1,1616
Mexican,295,332,264,162,57,1,1111
Latin American,208,215,470,188,14,1,1096
Japanese,561,240,187,31,45,1,1065
Italian,599,184,111,39,65,2,1000
Bakery Products/Desserts,310,227,269,84,32,2,924


**What we found:**

1. **Manhattan** leads 6 of the 10 cuisines: American (2,470), Coffee/Tea (1,113), Italian (599), Japanese (561), Pizza (484) and Bakery Products/Desserts (310).
2. **Queens** leads Chinese (765) and Latin American (470).
3. **Brooklyn** leads Caribbean (346) and Mexican (332), and is second in many others.
4. **Staten Island** has the fewest in 8 of the 10 cuisines. The exception is the Bronx, which is lowest for Japanese (31 vs 45) and Italian (39 vs 65). (The Unknown column is not counted here.)

**Why it matters:** each borough has a different mix of cuisines. If cuisine is linked to inspection scores, part of a borough's average score could come from its food types and not only from the borough itself. We need to keep this in mind when we compare boroughs later.

#### Inspection scores

**What we want to know:** how many inspections have a good score and how many have a bad one. The score is the total of violation points, so **a lower score is better**. We use the score and not the letter grade, because many inspections have a score but no grade.

| Score (violation points) | Letter grade it earns |
|---|---|
| 0 to 13 | A (best) |
| 14 to 27 | B |
| 28 or more | C (worst) |

**Why some inspections have no grade:** according to the NYC Health Department guide, an inspection that does not earn an A on the first visit is scored but not graded. The restaurant is re-inspected, typically within a month, and that re-inspection is graded.

**How we do it:** each inspection has several rows (one per violation) and the score repeats on each of them. So we first keep one row per inspection, then we drop inspections with no score, and we count how many fall in each range.

**Source:** NYC Health Department, [How We Score and Grade](https://www.nyc.gov/assets/doh/downloads/pdf/rii/restaurant-grading-faq.pdf) 

#### Part 2: Counting inspections by score group

**Intention:** I want to put each inspection into a score group (A, B or C range) and count how many inspections are in each group.

- The data has one row for each violation, so I keep only one row per inspection. If I don't, I would count the same inspection many times.
- I remove the inspections that have no score, because I can't put them in a group.
- I make three groups: 0 to 13, 14 to 27, and 28 or more.
- The `labels` line only gives a name to each group. It does not change any numbers.

In [38]:
inspections = inspected.drop_duplicates(subset=["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"])
scores = inspections["SCORE"].dropna()

bands = pd.cut(scores, bins=[-1, 13, 27, float("inf")],
               labels=["0 to 13 (A range)", "14 to 27 (B range)", "28 or more (C range)"])

score_table = pd.DataFrame({
    "inspections": bands.value_counts().sort_index(),
    "percent": (bands.value_counts(normalize=True).sort_index() * 100).round(1)
})

print("Inspections:", len(inspections))
print("Inspections with a score:", len(scores))
print("Perfect score (0):", (scores == 0).sum())
score_table

Inspections: 93292
Inspections with a score: 82956
Perfect score (0): 2097


,inspections,percent
SCORE,,
0 to 13 (A range),49670,59.9
14 to 27 (B range),18018,21.7
28 or more (C range),15268,18.4


**What we found:**

1. I found 93,292 inspections in total. 82,956 of them have a score, so 10,336 (about 11%) have no score and are not counted in the groups.
2. Most inspections are in the A range: 49,670 (59.9%), almost 6 out of 10.
3. The B range has 18,018 inspections (21.7%) and the C range has 15,268 (18.4%).
4. 2,097 inspections got a perfect score of 0.

**Why it matters:** about 4 out of 10 scored inspections are in the B or C range, so there is a good amount of variation to analyze. I used the score and not the letter grade, so I did not lose the inspections that have a score but no grade. I counted all inspection types together, so we should check later if some types have higher scores than others. **I also need to run this again after cleaning to see if the numbers change.**

#### Task 3: Source-row ID

**Intention:** I want to give every row a number that shows where it was in the original file. I do it now, before I sort or remove anything, so I can always trace a row back to the raw data.

- I add the new column `SOURCE_ROW_ID` only to the working copy (`df_work`). The raw data (`df`) stays untouched.
- Row 1 is the first data row of the file.

In [39]:
df_work = df.copy()
df_work.insert(0, "SOURCE_ROW_ID", range(1, len(df_work) + 1))

print("Rows:", len(df_work))
print("Columns:", df_work.shape[1], "(raw df has", df.shape[1], ")")
print("Unique IDs:", df_work["SOURCE_ROW_ID"].nunique())
print("First ID:", df_work["SOURCE_ROW_ID"].min(), "| Last ID:", df_work["SOURCE_ROW_ID"].max())
df_work.head(3)

Rows: 296012
Columns: 28 (raw df has 27 )
Unique IDs: 296012
First ID: 1 | Last ID: 296012


,SOURCE_ROW_ID,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,SCORE,GRADE,GRADE DATE,RECORD DATE,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
0,1,50181368,FROYOLICIOUS,Staten Island,2955,VETERANS ROAD WEST,10309.0,9177836601,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/26/2026,NaN,40.527800,-74.232641,503.0,51.0,22600.0,5150131.0,5.075110e+09,SI11,POINT (-74.23264118855 40.527800352215)
1,2,41395379,CITI FIELD STAND 114,Queens,000,126th & ROOSEVELT AVENUE,NaN,7185958100,American,09/20/2024,No violations were recorded at the time of thi...,NaN,NaN,Not Applicable,0.0,A,09/20/2024,08/26/2026,Cycle Inspection / Initial Inspection,0.000000,0.000000,NaN,NaN,NaN,NaN,4.000000e+00,NaN,NaN
2,3,50169220,HERITAGE DELI,Queens,56-27,58 STREET,11378.0,7187915831,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/26/2026,NaN,40.724601,-73.912450,402.0,26.0,21900.0,4059724.0,4.026640e+09,QN31,POINT (-73.912449613268 40.724601345342)


In [40]:
## show all columns and the first 11 rows
pd.set_option("display.max_columns", None)
df_work.head(11)

,SOURCE_ROW_ID,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,SCORE,GRADE,GRADE DATE,RECORD DATE,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
0,1,50181368,FROYOLICIOUS,Staten Island,2955,VETERANS ROAD WEST,10309.0,9177836601,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/26/2026,NaN,40.527800,-74.232641,503.0,51.0,22600.0,5150131.0,5.075110e+09,SI11,POINT (-74.23264118855 40.527800352215)
1,2,41395379,CITI FIELD STAND 114,Queens,000,126th & ROOSEVELT AVENUE,NaN,7185958100,American,09/20/2024,No violations were recorded at the time of thi...,NaN,NaN,Not Applicable,0.0,A,09/20/2024,08/26/2026,Cycle Inspection / Initial Inspection,0.000000,0.000000,NaN,NaN,NaN,NaN,4.000000e+00,NaN,NaN
2,3,50169220,HERITAGE DELI,Queens,56-27,58 STREET,11378.0,7187915831,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/26/2026,NaN,40.724601,-73.912450,402.0,26.0,21900.0,4059724.0,4.026640e+09,QN31,POINT (-73.912449613268 40.724601345342)
3,4,50148845,MAKE MY PASTRY,Queens,40-05,SKILLMAN AVENUE,11104.0,3479385678,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/26/2026,NaN,40.747196,-73.923417,402.0,26.0,17100.0,4002375.0,4.001830e+09,QN31,POINT (-73.923416505935 40.747195864681)
4,5,50162734,IQ FOODS GROUP LLC,Manhattan,1,HERALD SQUARE,10001.0,3472440804,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/26/2026,NaN,40.750542,-73.987541,105.0,4.0,10900.0,1089436.0,1.008100e+09,MN17,POINT (-73.987541021756 40.750541907818)
5,6,50179812,KAMPOS TAVERNA,Brooklyn,8413,3 AVENUE,11209.0,3478570516,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/26/2026,NaN,40.624842,-74.030480,310.0,47.0,6200.0,3152742.0,3.060250e+09,BK31,POINT (-74.030479756047 40.624841632068)
6,7,50189901,MARCO VACCHI,Manhattan,646,9 AVENUE,10036.0,3477811121,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/26/2026,NaN,40.760572,-73.991051,104.0,3.0,12100.0,1025004.0,1.010360e+09,MN15,POINT (-73.991051426489 40.760571586789)
7,8,41194745,GOLDEN PALACE GOURMET,Queens,14009,CHERRRY AVENUE,NaN,7188864383,Chinese,12/09/2024,Violations were cited in the following area(s).,04M,Live roaches in facility's food or non-food area.,Critical,28.0,NaN,NaN,08/26/2026,Cycle Inspection / Initial Inspection,0.000000,0.000000,NaN,NaN,NaN,NaN,4.000000e+00,NaN,NaN
8,9,50168503,LA QUINTA INN AND SUITES CENTRAL PARK,Manhattan,31,WEST 71 STREET,10023.0,5169159068,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/26/2026,NaN,40.776179,-73.977983,107.0,6.0,15700.0,1028614.0,1.011240e+09,MN14,POINT (-73.977982557122 40.776179253854)
9,10,50178739,WOOPS MACARONS,Manhattan,151,WEST 34 STREET,10001.0,9175876070,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,08/26/2026,NaN,40.750509,-73.989479,105.0,3.0,10900.0,1015219.0,1.008100e+09,MN17,POINT (-73.989479170444 40.750509164767)


In [41]:
print("Rows with an empty ID:", df_work["SOURCE_ROW_ID"].isna().sum())

Rows with an empty ID: 0


**What we found:** I added `SOURCE_ROW_ID` to the working copy. It has 296,012 unique values from 1 to 296,012, the same as the number of rows, and no row has an empty ID. The raw data (`df`) was not changed, so it still has 27 columns.

**Why it matters:** after I sort or remove duplicates, I can use this ID to find any row in the original file. We will also use it in the duplicate exclusion log.

#### Task 4: Review column names and data types

**Intention:** I want to document the original structure of the data: every column, its current type, how many cells are empty, and how many different values it has. I also write what type each column should become later.

I do not change any column names in this step, so I don't need a mapping between original and new names. If we rename columns later, I will add a mapping table.

In [42]:
column_info = pd.DataFrame({
    "column": df_work.columns,
    "current_type": df_work.dtypes.astype(str).values,
    "empty_cells": df_work.isna().sum().values,
    "different_values": df_work.nunique().values,
})
column_info["empty_percent"] = (column_info["empty_cells"] / len(df_work) * 100).round(1)

plan_text = ["ZIPCODE", "Census Tract", "BIN", "BBL", "Community Board", "Council District"]
plan_check = ["CAMIS"]
plan_date = ["INSPECTION DATE", "GRADE DATE", "RECORD DATE"]

def planned(c):
    if c in plan_text:
        return "text (keep leading zeros), Task 5"
    if c in plan_check:
        return "check format, Task 5"
    if c in plan_date:
        return "date, Task 7"
    return "keep as is"

column_info["planned_type"] = column_info["column"].apply(planned)

print("Original column names kept unchanged:", list(df.columns) == list(df_work.columns[1:]))
column_info

Original column names kept unchanged: True


,column,current_type,empty_cells,different_values,empty_percent,planned_type
0,SOURCE_ROW_ID,int64,0,296012,0.0,keep as is
1,CAMIS,int64,0,31366,0.0,"check format, Task 5"
2,DBA,object,2,24628,0.0,keep as is
3,BORO,object,0,6,0.0,keep as is
4,BUILDING,object,983,8558,0.3,keep as is
5,STREET,object,35,2311,0.0,keep as is
6,ZIPCODE,float64,3082,256,1.0,"text (keep leading zeros), Task 5"
7,PHONE,object,98,27705,0.0,keep as is
8,CUISINE DESCRIPTION,object,3794,90,1.3,keep as is
9,INSPECTION DATE,object,0,1966,0.0,"date, Task 7"


**What we found:**

1. **The data has 28 columns:** 2 whole-number columns (`SOURCE_ROW_ID`, `CAMIS`), 9 decimal columns and 17 text columns. I did not change any column names.
2. **9 decimal columns hold identifiers or codes:** `ZIPCODE`, `Community Board`, `Council District`, `Census Tract`, `BIN` and `BBL` should be text later. `Latitude`, `Longitude` and `SCORE` are real numbers, so they stay as decimals.
3. **The three date columns** (`INSPECTION DATE`, `GRADE DATE`, `RECORD DATE`) are stored as text, so I convert them in Task 7.
4. **`GRADE` is empty in 150,672** **rows (50.9%) and `GRADE DATE` in 160,961 rows (54.4%)**. `SCORE` is empty in 17,226 rows (5.8%). This is why I use the score and not the grade.
5. **Some columns have exactly the same number of empty cells:** `ACTION` and `INSPECTION TYPE` (3,714 each), `VIOLATION CODE` and `VIOLATION DESCRIPTION` (6,444 each), and `Latitude`, `Longitude` and `BBL` (1,672 each). They may be empty in the same rows. I have not checked this yet.
6. `RECORD DATE` has only 1 different value in all rows.

**Why it matters:** Now, we know which columns need type fixes before analysis, and Trinity can use this table as the starting point for the data dictionary.

#### Task 5 Correct identifier formats

**Part 1.Intention:** I want to see how the identifier columns (`CAMIS`, `ZIPCODE`, `Community Board`, `Council District`, `Census Tract`, `BIN`, `BBL`) are written in the original file, before I change anything. I read them again from the CSV as text, so pandas does not turn them into decimals. I only look in this step. I do not change any data.

In [43]:
id_cols = ["CAMIS", "ZIPCODE", "Community Board", "Council District", "Census Tract", "BIN", "BBL"]
raw_ids = pd.read_csv("data/DOHMH_New_York_City_Restaurant_Inspection_Results_20260827.csv", usecols=id_cols, dtype=str, low_memory=False)

rows = []
for c in id_cols:
    s = raw_ids[c].dropna()
    rows.append({
        "column": c,
        "filled": len(s),
        "empty": int(raw_ids[c].isna().sum()),
        "digit_lengths": sorted(s.str.len().unique().tolist()),
        "not_only_digits": int((~s.str.fullmatch(r"\d+")).sum()),
        "example": s.iloc[0],
    })

id_review = pd.DataFrame(rows)
id_review

,column,filled,empty,digit_lengths,not_only_digits,example
0,CAMIS,296012,0,[8],0,50181368
1,ZIPCODE,292930,3082,[5],0,10309
2,Community Board,291302,4710,[3],0,503
3,Council District,291270,4742,[2],0,51
4,Census Tract,291270,4742,[6],0,022600
5,BIN,290066,5946,[7],0,5150131
6,BBL,294340,1672,"[1, 10]",0,5075110001


**Part.2** I want to store `ZIPCODE`, `Community Board`, `Council District`, `Census Tract`, `BIN` and `BBL` as text in the working copy, using the original text from the file, so the leading zeros are kept. Empty cells stay empty. I do not touch `CAMIS`, because it already has a clean 8 digit format. I also check that the numbers are the same as before and I count the `BBL` values that do not have 10 digits. I do not change or delete those values here.

In [44]:
text_cols = ["ZIPCODE", "Community Board", "Council District", "Census Tract", "BIN", "BBL"]

for c in text_cols:
    df_work[c] = raw_ids[c]

check = pd.DataFrame({
    "type": df_work[text_cols].dtypes.astype(str),
    "empty_now": df_work[text_cols].isna().sum(),
    "empty_before": df[text_cols].isna().sum(),
})
check["same_numbers"] = [
    bool((df_work[c].dropna().astype(float).values == df[c].dropna().values).all())
    for c in text_cols
]
display(check)

bbl_len = df_work["BBL"].str.len()
odd_bbl = bbl_len.notna() & (bbl_len != 10)
print("BBL values that are not 10 digits:", odd_bbl.sum())
df_work.loc[odd_bbl, "BBL"].value_counts()

,type,empty_now,empty_before,same_numbers
ZIPCODE,object,3082,3082,True
Community Board,object,4710,4710,True
Council District,object,4742,4742,True
Census Tract,object,4742,4742,True
BIN,object,5946,5946,True
BBL,object,1672,1672,True


BBL values that are not 10 digits: 4274


BBL
1    2048
4     917
3     582
2     563
5     164
Name: count, dtype: int64

**What we found:**

1. I stored `ZIPCODE`, `Community Board`, `Council District`, `Census Tract`, `BIN` and `BBL` as text in the working copy. The empty cells did not change, and the numbers are the same as in the decimal version, so only the type changed. `CAMIS` was already clean (8 digits), so I left it as it is.
2. 4,274 `BBL` values are not 10 digits. They are all one single digit: 1 (2,048 times), 4 (917), 3 (582), 2 (563) and 5 (164). I did not change or delete them.
3. These single digits look like incomplete BBL codes, not real ones. This is my guess. I need to check it in Task 10, for example by comparing them with the borough of each restaurant.

**Why it matters:** the identifiers now keep their leading zeros (like the `022600` census tract), and we can join on them without problems later. The 4,274 incomplete `BBL` values are an open issue. I will check and fix them in Task 10, together with the other geographic columns.

#### Task 6: Standardize basic text formatting - Look for extra spaces and empty strings

I want to see which text columns have extra spaces at the start or the end of a value, and which have values that are empty or only spaces. I only look in this step. I do not change any data. A truly empty cell (missing value) is a different thing, and I keep those as they are.

**What it does:** for every text column, it counts the values that have spaces at the start or the end  "Pizza" and "Pizza " would be counted as two different values in any analysis, and the values that are empty or only spaces (like " "). Nothing is changed. Also even though they are the same thing. We only fix what the table shows that really needs fixing.

In [45]:
text_columns = df_work.select_dtypes(include="object").columns

rows = []
for c in text_columns:
    s = df_work[c].dropna().astype(str)
    rows.append({
        "column": c,
        "filled": len(s),
        "spaces_at_start_or_end": int((s != s.str.strip()).sum()),
        "empty_or_only_spaces": int((s.str.strip() == "").sum()),
    })

text_review = pd.DataFrame(rows)
text_review

,column,filled,spaces_at_start_or_end,empty_or_only_spaces
0,DBA,296010,0,0
1,BORO,296012,0,0
2,BUILDING,295029,0,0
3,STREET,295977,0,0
4,ZIPCODE,292930,0,0
5,PHONE,295914,0,0
6,CUISINE DESCRIPTION,292218,0,0
7,INSPECTION DATE,296012,0,0
8,ACTION,292298,0,0
9,VIOLATION CODE,289568,0,0


**What we found:**

1. I checked all 23 text columns (the 17 original text columns and the 6 identifiers I stored as text in Task 5).
2. None of them has spaces at the start or the end of a value, and none has values that are only spaces.
3. Truly empty fields in the file are already read as empty cells. I keep them as they are, and they are already counted in my Task 2 baseline.

**What I did:** nothing needed to change, so I did not trim or edit any value. The violation descriptions and category values are exactly as in the original file.


#### Task 7: Convert and validate dates

**Part 1. Intention:** I want to test if the three date columns (`INSPECTION DATE`, `GRADE DATE`, `RECORD DATE`) can be turned into real dates in the format month/day/year. I count how many values convert and how many fail. I also see the earliest and the latest date of each column. I only test in this step. I do not change any data. An empty date is a missing value, not a failure.

In [46]:
date_cols = ["INSPECTION DATE", "GRADE DATE", "RECORD DATE"]

rows = []
for c in date_cols:
    s = df_work[c]
    parsed = pd.to_datetime(s, format="%m/%d/%Y", errors="coerce")
    rows.append({
        "column": c,
        "filled": int(s.notna().sum()),
        "converted_ok": int(parsed.notna().sum()),
        "failed_to_convert": int((s.notna() & parsed.isna()).sum()),
        "earliest": parsed.min(),
        "latest": parsed.max(),
    })

date_test = pd.DataFrame(rows)
date_test

,column,filled,converted_ok,failed_to_convert,earliest,latest
0,INSPECTION DATE,296012,296012,0,1900-01-01,2026-08-24
1,GRADE DATE,135051,135051,0,2011-08-10,2026-08-24
2,RECORD DATE,296012,296012,0,2026-08-26,2026-08-26


**What we found:**

1. All three date columns can be converted to real dates in the format month/day/year. None failed (0 failures in all three).
2. `INSPECTION DATE` goes from 1900-01-01 to 2026-08-24. The 1900 date is the placeholder for restaurants not inspected yet, and I handle it in Task 8.
3. `GRADE DATE` goes from 2011-08-10 to 2026-08-24, and it is empty in 160,961 rows, as I saw in before (taks 4)
4. `RECORD DATE` has one single value, 2026-08-26, in every row. My guess is that it is the date the file was created. I have not confirmed it.
5. The latest dates are all before the snapshot date (2026-08-27). I check this properly with a flag in the next step.

**Why it matters:** the format is the same everywhere, so I can convert safely without losing any date.

**Part 2. Intention:** I want to convert `INSPECTION DATE`, `GRADE DATE` and `RECORD DATE` into real dates in the working copy. Empty dates stay empty. I do not change the 1900-01-01 placeholder dates here, because I handle them in Task 8. Then I check that the type changed and that the number of empty dates did not.

In [47]:
for c in date_cols:
    df_work[c] = pd.to_datetime(df_work[c], format="%m/%d/%Y", errors="coerce")

check_dates = pd.DataFrame({
    "type": df_work[date_cols].dtypes.astype(str),
    "empty_now": df_work[date_cols].isna().sum(),
    "empty_before": df[date_cols].isna().sum(),
})
check_dates

,type,empty_now,empty_before
INSPECTION DATE,datetime64[ns],0,0
GRADE DATE,datetime64[ns],160961,160961
RECORD DATE,datetime64[ns],0,0


**What we found:**

1. I converted `INSPECTION DATE`, `GRADE DATE` and `RECORD DATE` into real dates in the working copy.
2. The number of empty dates did not change: 0 in `INSPECTION DATE`, 160,961 in `GRADE DATE` and 0 in `RECORD DATE`. So no value was lost.
3. I did not change the 1900-01-01 placeholder dates. I handle them in Task 8.

**Why it matters:** I can now sort by date and compare dates. The original text is still in the raw data (`df`), and `SOURCE_ROW_ID` links each row to it.

**Part. 3** I want to check if any date is after the snapshot date (2026-08-27), if any grade date is before its inspection date, and if any inspection date is after the record date. I also look at how many days there are between the inspection date and the grade date. These are my own rules for what looks suspicious. In this step I only count. If I find rows, I will flag them in the next step.

In [48]:
SNAPSHOT_DATE = pd.Timestamp("2026-08-27")

print("Empty dates (same as before conversion?):")
print(df_work[date_cols].isna().sum().to_string())

date_checks = {
    "any date after the snapshot date": int(((df_work[date_cols] > SNAPSHOT_DATE).any(axis=1)).sum()),
    "grade date before inspection date": int((df_work["GRADE DATE"] < df_work["INSPECTION DATE"]).sum()),
    "inspection date after record date": int((df_work["INSPECTION DATE"] > df_work["RECORD DATE"]).sum()),
}
display(pd.Series(date_checks, name="rows").to_frame())

days_to_grade = (df_work["GRADE DATE"] - df_work["INSPECTION DATE"]).dt.days
print("Rows with both an inspection date and a grade date:", days_to_grade.notna().sum())
days_to_grade.value_counts().head()

Empty dates (same as before conversion?):
INSPECTION DATE         0
GRADE DATE         160961
RECORD DATE             0


,rows
any date after the snapshot date,0
grade date before inspection date,0
inspection date after record date,0


Rows with both an inspection date and a grade date: 135051


0.0    135051
Name: count, dtype: int64

**What we found:**

1. No date is after the snapshot date (2026-08-27).
2. No grade date is before its inspection date, and no inspection date is after the record date (0 rows in each check).
3. In all 135,051 rows that have both an inspection date and a grade date, the two dates are the same day (0 days difference).
4. I did not find anything to flag, so I did not add flag columns.

**Why it matters:** the dates are consistent with each other, so I can use them for sorting and for comparing inspections over time. My checks only cover these three rules. I did not check other kinds of odd dates.

**Part 4. Intention:** I want to check if `GRADE` and `GRADE DATE` are filled together. I count the rows that have a grade but no grade date, and the rows that have a grade date but no grade. I only count in this step.

In [49]:
print("Has a grade but no grade date:", (df_work["GRADE"].notna() & df_work["GRADE DATE"].isna()).sum())
print("Has a grade date but no grade:", (df_work["GRADE"].isna() & df_work["GRADE DATE"].notna()).sum())

no_date = df_work["GRADE"].notna() & df_work["GRADE DATE"].isna()
df_work.loc[no_date, "GRADE"].value_counts()

Has a grade but no grade date: 10289
Has a grade date but no grade: 0


GRADE
N    10289
Name: count, dtype: int64

**What we found:**

1. 10,289 rows have a grade but no grade date. All of them have the grade `N`.
2. 0 rows have a grade date but no grade.
3. The empty `GRADE DATE` cells (160,961) are exactly the rows with no grade (150,672) plus these rows with grade `N` (10,289).
4. My reading is that `N` means "Not Yet Graded", so there is no date because no grade was given yet. This is my interpretation, based on how the NYC guide describes this status.

**Why it matters:** an empty `GRADE DATE` is not a mistake in the data. It follows the grade: no grade, or grade `N`. I do not need to fill or fix those cells.

#### Task 8: Identify placeholder inspection records

**Step 1. Intention:** I want to look at the rows with the placeholder inspection date (1900-01-01), which mean that the restaurant has not been inspected yet. I count them, I check how many different restaurants they belong to, and I compare which columns are empty in these rows against the other rows. I only look in this step. I do not change or delete anything.

In [50]:
PLACEHOLDER_DATE = pd.Timestamp("1900-01-01")
is_placeholder = df_work["INSPECTION DATE"] == PLACEHOLDER_DATE

placeholder_camis = df_work.loc[is_placeholder, "CAMIS"].unique()
also_real = df_work.loc[~is_placeholder & df_work["CAMIS"].isin(placeholder_camis), "CAMIS"].nunique()

print("Placeholder rows:", is_placeholder.sum())
print("Different restaurants in those rows:", len(placeholder_camis))
print("Of those, restaurants that also have rows with a real inspection date:", also_real)

cols = ["ACTION", "INSPECTION TYPE", "VIOLATION CODE", "SCORE", "GRADE", "GRADE DATE", "CUISINE DESCRIPTION"]
compare = pd.DataFrame({
    "empty_in_placeholder_rows": df_work.loc[is_placeholder, cols].isna().sum(),
    "empty_in_other_rows": df_work.loc[~is_placeholder, cols].isna().sum(),
})
compare

Placeholder rows: 3714
Different restaurants in those rows: 3714
Of those, restaurants that also have rows with a real inspection date: 0


,empty_in_placeholder_rows,empty_in_other_rows
ACTION,3714,0
INSPECTION TYPE,3714,0
VIOLATION CODE,3714,2730
SCORE,3714,13512
GRADE,3714,146958
GRADE DATE,3714,157247
CUISINE DESCRIPTION,3714,80


**Step 2 Intention:** I want to add a column called `AWAITING_INSPECTION` that is True for the placeholder rows (1900-01-01) and False for real inspections. I keep all the rows and I keep the original date, so nothing is deleted. Then I check that the number of rows did not change and that the counts add up.

In [51]:
df_work["AWAITING_INSPECTION"] = df_work["INSPECTION DATE"] == PLACEHOLDER_DATE

print("Total rows:", len(df_work), "| Columns:", df_work.shape[1])
print("Flagged (awaiting inspection):", df_work["AWAITING_INSPECTION"].sum())
print("Not flagged (real inspections):", (~df_work["AWAITING_INSPECTION"]).sum())
print("Restaurants with real inspections:", df_work.loc[~df_work["AWAITING_INSPECTION"], "CAMIS"].nunique())

Total rows: 296012 | Columns: 29
Flagged (awaiting inspection): 3714
Not flagged (real inspections): 292298
Restaurants with real inspections: 27652


**What we found:**

1. I added the column `AWAITING_INSPECTION`. It is True in 3,714 rows (the placeholder date 1900-01-01) and False in 292,298 rows (real inspections).
2. The working copy still has all 296,012 rows, and now 29 columns. I did not delete any row or change any date.
3. The restaurants with real inspections are 27,652, the same number as in my cuisine count. The other 3,714 restaurants are only awaiting their first inspection.

**Why it matters:** from now on, I can separate real inspections from placeholders with one column. For any analysis about inspections, scores or grades, I use only the rows where `AWAITING_INSPECTION` is False.

#### Task 9: Review exact duplicates

**** I want to look at the rows that are exact copies of another row, using only the original 27 columns. I leave out `SOURCE_ROW_ID` and my flag column, because they are new and would make every row look different. I count the duplicates, check how they are grouped, and look at examples. I only look in this step. I do not remove anything.

In [52]:
orig_cols = list(df.columns)

extra_copies = df_work.duplicated(subset=orig_cols, keep="first")
all_copies = df_work.duplicated(subset=orig_cols, keep=False)

print("Extra copies (rows I would remove):", extra_copies.sum())
print("Rows involved in a duplicate group (originals + copies):", all_copies.sum())
print("Different restaurants involved:", df_work.loc[all_copies, "CAMIS"].nunique())
print("Duplicate rows that are placeholder rows:", df_work.loc[all_copies, "AWAITING_INSPECTION"].sum())

print("\nHow many rows in each group (size : number of groups):")
group_sizes = df_work[all_copies].groupby(orig_cols, dropna=False).size()
print(group_sizes.value_counts().to_string())

show_cols = ["SOURCE_ROW_ID", "CAMIS", "DBA", "INSPECTION DATE", "VIOLATION CODE", "SCORE"]
df_work.loc[all_copies, show_cols].sort_values(["CAMIS", "INSPECTION DATE", "VIOLATION CODE", "SOURCE_ROW_ID"]).head(10)

Extra copies (rows I would remove): 169
Rows involved in a duplicate group (originals + copies): 338
Different restaurants involved: 32
Duplicate rows that are placeholder rows: 0

How many rows in each group (size : number of groups):
2    169


,SOURCE_ROW_ID,CAMIS,DBA,INSPECTION DATE,VIOLATION CODE,SCORE
76532,76533,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,2026-02-10,02G,23.0
79969,79970,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,2026-02-10,02G,23.0
175200,175201,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,2026-02-10,06B,23.0
175389,175390,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,2026-02-10,06B,23.0
75354,75355,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,2026-02-10,09A,23.0
77497,77498,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,2026-02-10,09A,23.0
63510,63511,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,2026-02-10,10F,23.0
66324,66325,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,2026-02-10,10F,23.0
64186,64187,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,2026-02-10,10G,23.0
67238,67239,40390409,THE FAMOUS JIMBO'S HAMBURGER PALACE,2026-02-10,10G,23.0


**What we found:**

1. I found 169 extra copies of rows, which matches my Task 2 baseline. With their originals, 338 rows are involved, and every duplicate group has exactly 2 rows.
2. The duplicates belong to only 32 restaurants, and none of them is a placeholder row.
3. The two copies are not next to each other in the file. For example, in the Jimbo's inspection of 2026-02-10, the same violation codes (like 02G and 10F) appear twice, in rows far apart. My reading is that the same inspection was entered twice.

**Why it matters:** if I count both copies, the violations of those inspections are counted twice. I only compared the 27 original columns, because `SOURCE_ROW_ID` is different in every row.

**Part 2. Intention** I want to write down my rule for duplicates and build the exclusion log before I remove anything. My rule: a row is an exact duplicate if all 27 original columns are the same as an earlier row. I keep the first row (lowest `SOURCE_ROW_ID`) and I log the later copies. First I check if the same violation code is ever repeated inside the same inspection for another reason, because then removing copies would not be safe.

In [53]:
# 1. Is a violation code ever repeated in the same inspection, apart from exact copies?
key = ["CAMIS", "INSPECTION DATE", "INSPECTION TYPE", "VIOLATION CODE"]
same_key = df_work.duplicated(subset=key, keep=False) & df_work["VIOLATION CODE"].notna()
print("Rows that repeat a violation code in the same inspection:", same_key.sum())
print("Of those, rows that are exact copies:", (same_key & all_copies).sum())

# 2. Exclusion log (only the rows I would remove)
sub = df_work.loc[all_copies]
sub_first_id = sub.groupby(orig_cols, dropna=False)["SOURCE_ROW_ID"].transform("min")
is_extra = extra_copies.loc[sub.index]

exclusion_log = sub.loc[is_extra, ["SOURCE_ROW_ID", "CAMIS", "DBA", "INSPECTION DATE", "VIOLATION CODE"]].copy()
exclusion_log["KEPT_SOURCE_ROW_ID"] = sub_first_id[is_extra]
exclusion_log["REASON"] = "exact duplicate of an earlier row (all 27 original columns equal)"

print("Rows in the exclusion log:", len(exclusion_log))
print("Every removed row comes after the row I keep:", (exclusion_log["SOURCE_ROW_ID"] > exclusion_log["KEPT_SOURCE_ROW_ID"]).all())
exclusion_log.head(5)

Rows that repeat a violation code in the same inspection: 338
Of those, rows that are exact copies: 338
Rows in the exclusion log: 169
Every removed row comes after the row I keep: True


,SOURCE_ROW_ID,CAMIS,DBA,INSPECTION DATE,VIOLATION CODE,KEPT_SOURCE_ROW_ID,REASON
2527,2528,50125107,LEE'S DELI,2026-02-10,02B,1151,exact duplicate of an earlier row (all 27 orig...
3561,3562,50185382,TOUS LES JOURS,2026-06-29,02B,1505,exact duplicate of an earlier row (all 27 orig...
4340,4341,50178594,HIGH BEAM COFFEE ROASTERY,2026-02-10,10B,4232,exact duplicate of an earlier row (all 27 orig...
7564,7565,40537290,NORA'S PARK BENCH CAFE,2026-03-21,06C,5787,exact duplicate of an earlier row (all 27 orig...
8427,8428,41249363,ROASTING PLANT COFFEE,2026-02-10,10B,5673,exact duplicate of an earlier row (all 27 orig...


**What we found:**

1. My rule: a row is an exact duplicate if all 27 original columns are the same as an earlier row. I keep the first row (lowest `SOURCE_ROW_ID`) and I log the later copy.
2. I checked the rule. 338 rows repeat a violation code in the same inspection, and all 338 are exact copies. So there is no case where a violation is repeated in an inspection for another reason, and removing the copies does not remove real information.
3. I built an exclusion log with 169 rows. Each row has the `SOURCE_ROW_ID` of the removed copy and the `SOURCE_ROW_ID` of the row I keep. Every removed row comes after the row I keep.

**Why it matters:** the log lets anyone trace each removed row, and the copies are identical, so it does not matter which one I keep.

**Part 3. Intention::** I want to remove the 169 copies that are in my exclusion log from the working copy. I remove them using the `SOURCE_ROW_ID` in the log. Then I check the row counts before and after, that no exact duplicate is left, and that the restaurant count did not change. The raw data is not changed, and the log keeps a record of each removed row.

In [54]:
rows_before = len(df_work)
restaurants_before = df_work["CAMIS"].nunique()

df_work = df_work[~df_work["SOURCE_ROW_ID"].isin(exclusion_log["SOURCE_ROW_ID"])].copy()

print("Rows before:", rows_before)
print("Rows removed:", rows_before - len(df_work))
print("Rows after:", len(df_work))
print("Rows in the exclusion log:", len(exclusion_log))
print("Restaurants before and after:", restaurants_before, "and", df_work["CAMIS"].nunique())
print("Exact duplicates left:", df_work.duplicated(subset=orig_cols).sum())
print("SOURCE_ROW_ID still unique:", df_work["SOURCE_ROW_ID"].is_unique)

Rows before: 296012
Rows removed: 169
Rows after: 295843
Rows in the exclusion log: 169
Restaurants before and after: 31366 and 31366
Exact duplicates left: 0
SOURCE_ROW_ID still unique: True


**What we found:**

1. I removed the 169 copies listed in my exclusion log from the working copy. The rows went from 296,012 to 295,843.
2. The number of restaurants did not change (31,366 before and after), and no exact duplicates are left.
3. `SOURCE_ROW_ID` is still unique, so every remaining row can be traced to the original file. The raw data (`df`) was not changed.

**Why it matters:** the violations of the duplicated inspections are now counted once. The exclusion log keeps a record of every removed row, so this step can be checked or undone.

#### Task 10: Borough and geographic fields

**Part 1. Intention:** I want to look at the rows where `BORO` is 0, which means the borough is unknown. I count the rows and the restaurants, and I compare which location columns are empty in these rows against all the other rows. I only look in this step. I do not change any data.

In [55]:
print(df_work["BORO"].value_counts().to_string())

boro0 = df_work["BORO"] == "0"
print("\nRows with BORO = 0:", boro0.sum())
print("Different restaurants:", df_work.loc[boro0, "CAMIS"].nunique())
print("Of those rows, awaiting inspection:", df_work.loc[boro0, "AWAITING_INSPECTION"].sum())

geo_cols = ["BUILDING", "STREET", "ZIPCODE", "Latitude", "Longitude", "Community Board",
            "Council District", "Census Tract", "BIN", "BBL", "NTA", "Location"]
pd.DataFrame({
    "empty_in_BORO0_rows": df_work.loc[boro0, geo_cols].isna().sum(),
    "empty_in_other_rows": df_work.loc[~boro0, geo_cols].isna().sum(),
})

BORO
Manhattan        109608
Brooklyn          74873
Queens            74192
Bronx             27226
Staten Island      9593
0                   351

Rows with BORO = 0: 351
Different restaurants: 84
Of those rows, awaiting inspection: 52


,empty_in_BORO0_rows,empty_in_other_rows
BUILDING,35,948
STREET,35,0
ZIPCODE,44,3032
Latitude,351,1321
Longitude,351,1321
Community Board,351,4353
Council District,351,4385
Census Tract,351,4385
BIN,351,5589
BBL,351,1321


**What we found:**

1. 351 rows have `BORO` = 0 (unknown borough). They belong to 84 different restaurants, and 52 of those rows are awaiting-inspection rows.
2. In all 351 rows, `Latitude`, `Longitude`, `Community Board`, `Council District`, `Census Tract`, `BIN`, `BBL`, `NTA` and `Location` are empty. So I cannot find the borough from any of those columns.
3. `ZIPCODE` is filled in 307 of the 351 rows and `STREET` in 316. My decision is not to guess the borough from the address or the zip code, because that would be inventing a value that is not in the data.

**Why it matters:** these rows cannot be used in any analysis by borough. I keep them in the data and mark them as unknown, so they are not mistaken for a real borough.

**Part 2. Intention:** I want to add a column `BORO_UNKNOWN` that is True when `BORO` is 0 and False otherwise. I keep the original `BORO` value and all the rows. Then I check that the flag count is the same as the number of rows with `BORO` = 0 and that no row was lost.

In [56]:
df_work["BORO_UNKNOWN"] = df_work["BORO"] == "0"

print("Total rows:", len(df_work), "| Columns:", df_work.shape[1])
print("BORO_UNKNOWN is True:", df_work["BORO_UNKNOWN"].sum())
print("BORO values still the same:", sorted(df_work["BORO"].unique()))

Total rows: 295843 | Columns: 30
BORO_UNKNOWN is True: 351
BORO values still the same: ['0', 'Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']


**What we found:**

1. I added the column `BORO_UNKNOWN`. It is True in 351 rows (where `BORO` is 0) and False in all the others.
2. The working copy still has all 295,843 rows, and now 30 columns. The original `BORO` values did not change.

**Why it matters:** I can separate the unknown boroughs from the real ones with one column. For any analysis by borough, I use only the rows where `BORO_UNKNOWN` is False.

**Part.3 Intention:** I want to check the `Latitude` and `Longitude` columns. I count the rows where both are 0, because that is not a real location. I also check if the real coordinates (not empty and not 0) are inside a rough box around New York City. I only look in this step. I do not change any data.

In [57]:
lat = df_work["Latitude"]
lon = df_work["Longitude"]

zero_coords = (lat == 0) & (lon == 0)
print("Rows with latitude and longitude both 0:", zero_coords.sum())
print("Different restaurants in those rows:", df_work.loc[zero_coords, "CAMIS"].nunique())
print("Rows where only one of them is 0:", ((lat == 0) ^ (lon == 0)).sum())
print("Empty latitude / empty longitude:", lat.isna().sum(), "/", lon.isna().sum())

real = lat.notna() & ~zero_coords
inside_nyc = lat.between(40.4, 41.0) & lon.between(-74.3, -73.6)
print("Real coordinates (not empty, not 0):", real.sum())
print("Of those, outside the rough NYC box:", (real & ~inside_nyc).sum())

print("\nRows with 0 coordinates, by borough:")
df_work.loc[zero_coords, "BORO"].value_counts()

Rows with latitude and longitude both 0: 3032
Different restaurants in those rows: 447
Rows where only one of them is 0: 0
Empty latitude / empty longitude: 1672 / 1672
Real coordinates (not empty, not 0): 291139
Of those, outside the rough NYC box: 0

Rows with 0 coordinates, by borough:


BORO
Manhattan        1594
Queens            596
Bronx             456
Brooklyn          284
Staten Island     102
Name: count, dtype: int64

**What we found:**

1. 3,032 rows have `Latitude` and `Longitude` both equal to 0. They belong to 447 different restaurants. No row has only one of them at 0.
2. `Latitude` and `Longitude` are also empty in 1,672 rows each (a different group of rows).
3. The other 291,139 rows have real coordinates, and none of them is outside my rough box around New York City.
4. The rows with 0 coordinates are spread over all five boroughs: Manhattan 1,594, Queens 596, Bronx 456, Brooklyn 284 and Staten Island 102. None of them has an unknown borough.

**Why it matters:** 0, 0 is a point in the ocean, not a restaurant location. If I use these rows in a map, in a distance calculation or in an average, the result would be wrong.

**Part 4. Intention:** I want to flag the 3,032 rows where `Latitude` and `Longitude` are both 0, and set those two values to empty in the working copy, because 0, 0 is not a real location. The flag `ZERO_COORDINATES` keeps a record of which rows were changed, and the original zeros are still in the raw data. I keep all the rows. Then I check the counts.

In [58]:
df_work["ZERO_COORDINATES"] = (df_work["Latitude"] == 0) & (df_work["Longitude"] == 0)
df_work.loc[df_work["ZERO_COORDINATES"], ["Latitude", "Longitude"]] = float("nan")

print("Total rows:", len(df_work), "| Columns:", df_work.shape[1])
print("ZERO_COORDINATES is True:", df_work["ZERO_COORDINATES"].sum())
print("Zeros left in latitude or longitude:", ((df_work["Latitude"] == 0) | (df_work["Longitude"] == 0)).sum())
print("Empty latitude / longitude now:", df_work["Latitude"].isna().sum(), "/", df_work["Longitude"].isna().sum())
print("Latitude range:", df_work["Latitude"].min(), "to", df_work["Latitude"].max())
print("Longitude range:", df_work["Longitude"].min(), "to", df_work["Longitude"].max())

Total rows: 295843 | Columns: 31
ZERO_COORDINATES is True: 3032
Zeros left in latitude or longitude: 0
Empty latitude / longitude now: 4704 / 4704
Latitude range: 40.49956271029 to 40.91282232639
Longitude range: -74.24910133173 to -73.70092805781


**What we found:**

1. I added the column `ZERO_COORDINATES`. It is True in 3,032 rows, the rows where `Latitude` and `Longitude` were both 0.
2. In those rows, I set `Latitude` and `Longitude` to empty. Now no zeros are left, and the empty count is 4,704 in each column (the 1,672 that were already empty plus the 3,032 changed).
3. The remaining coordinates range from 40.50 to 40.91 in latitude and from -74.25 to -73.70 in longitude, which looks like New York City.
4. All the rows are still there (295,843), and the original zeros are still in the raw data.

**Why it matters:** the maps, distances and averages will not be pulled toward a point in the ocean. The flag tells which rows had no real location.

**Part. 5 Intention:** I want to look again at the `BBL` values that are not 10 digits, now that the duplicates are removed. I count them, I check if they also have zero coordinates, and I compare them with the borough of each restaurant. I also compare the first digit of the normal 10 digit `BBL` values with the borough. I only look in this step. I do not change any data.

In [59]:
bbl_len = df_work["BBL"].str.len()
odd_bbl = bbl_len.notna() & (bbl_len != 10)
valid_bbl = bbl_len == 10

print("BBL values that are not 10 digits:", odd_bbl.sum())
print("Different restaurants:", df_work.loc[odd_bbl, "CAMIS"].nunique())
print("Of those rows, also zero coordinates:", (odd_bbl & df_work["ZERO_COORDINATES"]).sum())

print("\n1-digit BBL values by borough:")
display(pd.crosstab(df_work.loc[odd_bbl, "BORO"], df_work.loc[odd_bbl, "BBL"]))

print("First digit of 10-digit BBL values by borough:")
display(pd.crosstab(df_work.loc[valid_bbl, "BORO"], df_work.loc[valid_bbl, "BBL"].str[0]))

BBL values that are not 10 digits: 4268
Different restaurants: 612
Of those rows, also zero coordinates: 2985

1-digit BBL values by borough:


BBL,1,2,3,4,5
BORO,,,,,
Bronx,0,563,0,0,0
Brooklyn,0,0,582,0,0
Manhattan,2048,0,0,0,0
Queens,0,0,0,911,0
Staten Island,0,0,0,0,164


First digit of 10-digit BBL values by borough:


BBL,1,2,3,4,5
BORO,,,,,
Bronx,0,26597,0,0,0
Brooklyn,0,0,74260,0,0
Manhattan,107030,0,0,0,0
Queens,0,0,0,72589,0
Staten Island,0,0,0,0,9427


**What we found:**

1. After removing the duplicates, 4,268 `BBL` values are not 10 digits. They belong to 612 different restaurants.
2. 2,985 of those rows also had zero coordinates, so most of them are rows where no exact location was found.
3. Each single-digit `BBL` matches the borough of the restaurant exactly: 1 is Manhattan, 2 is Bronx, 3 is Brooklyn, 4 is Queens and 5 is Staten Island.
4. The first digit of the normal 10-digit `BBL` values follows the same pattern, with no exception.

**Why it matters:** the single digit is only the borough code, so it is not a real `BBL`, and I already have the borough in `BORO`. My guess from Task 5 was correct, and now I checked it in the data.

In [60]:
df_work["BBL_INCOMPLETE"] = odd_bbl
df_work.loc[df_work["BBL_INCOMPLETE"], "BBL"] = float("nan")

print("Total rows:", len(df_work), "| Columns:", df_work.shape[1])
print("BBL_INCOMPLETE is True:", df_work["BBL_INCOMPLETE"].sum())
print("Empty BBL now:", df_work["BBL"].isna().sum())
print("BBL lengths now:", sorted(df_work["BBL"].dropna().str.len().unique()))

Total rows: 295843 | Columns: 32
BBL_INCOMPLETE is True: 4268
Empty BBL now: 5940
BBL lengths now: [np.int64(10)]


**What we found:**

1. I added the column `BBL_INCOMPLETE`. It is True in 4,268 rows, the rows where `BBL` was a single digit.
2. In those rows, I set `BBL` to empty. Now every `BBL` that is not empty has exactly 10 digits, and the empty count is 5,940 (the 1,672 that were already empty plus the 4,268 changed).
3. No row was deleted (295,843 rows). The original single digits are still in the raw data, and they only repeated the borough, which I keep in `BORO`.

**Why it matters:** the `BBL` column now only has real 10 digit values or empty cells, so I can use it to join with other property data without wrong values.

#### Task 11: Check consistency and prepare the handoff

**Part 1. Intention:** I want to check that the rows of the same inspection (same restaurant, date and type) agree with each other. An inspection has one row per violation, but the score, grade, borough and restaurant details should be the same in all its rows. I count how many inspections have different values in any of these columns. I skip the placeholder rows, because they are not real inspections. I only look in this step. I do not change any data.

In [61]:
real_rows = df_work[~df_work["AWAITING_INSPECTION"]]
inspection_key = ["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"]
grouped = real_rows.groupby(inspection_key)

print("Real inspections:", grouped.ngroups)

check_cols = ["SCORE", "GRADE", "GRADE DATE", "ACTION", "DBA", "BORO", "CUISINE DESCRIPTION", "ZIPCODE"]
consistency = pd.DataFrame(
    {"inspections_with_different_values": [int((grouped[c].nunique(dropna=False) > 1).sum()) for c in check_cols]},
    index=check_cols,
)
consistency

Real inspections: 93292


,inspections_with_different_values
SCORE,0
GRADE,0
GRADE DATE,0
ACTION,0
DBA,0
BORO,0
CUISINE DESCRIPTION,0
ZIPCODE,0


**What we found:**

1. After the cleaning, there are 93,292 real inspections (restaurant + date + type), the same number as in my score count.
2. I compared 8 columns inside each inspection: `SCORE`, `GRADE`, `GRADE DATE`, `ACTION`, `DBA`, `BORO`, `CUISINE DESCRIPTION` and `ZIPCODE`. In all of them, 0 inspections have different values.
3. The placeholder rows are not included, because they are not real inspections.

**Why it matters:** each inspection has one score, one grade and one set of restaurant details, so I can safely keep one row per inspection when I want to analyze scores, without mixing values.

**Part 2. Intention:** I want a table that compares the raw data with the prepared data: rows, columns, unique restaurants and real inspections. This shows exactly what changed during my cleaning, and I will save it as part of the handoff.

In [62]:
raw_inspections = df[df["INSPECTION DATE"] != "01/01/1900"].drop_duplicates(
    subset=["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"]).shape[0]

before_after = pd.DataFrame({
    "before (raw)": [len(df), df.shape[1], df["CAMIS"].nunique(), raw_inspections],
    "after (prepared)": [len(df_work), df_work.shape[1], df_work["CAMIS"].nunique(), grouped.ngroups],
}, index=["rows", "columns", "unique restaurants (CAMIS)", "real inspections"])
before_after["change"] = before_after["after (prepared)"] - before_after["before (raw)"]
before_after

,before (raw),after (prepared),change
rows,296012,295843,-169
columns,27,32,5
unique restaurants (CAMIS),31366,31366,0
real inspections,93292,93292,0


**What we found:**

| | Before (raw) | After (prepared) | Change |
|---|---|---|---|
| Rows | 296,012 | 295,843 | -169 |
| Columns | 27 | 32 | +5 |
| Unique restaurants (`CAMIS`) | 31,366 | 31,366 | 0 |
| Real inspections | 93,292 | 93,292 | 0 |

The 169 rows are the exact duplicates I removed. The 5 new columns are `SOURCE_ROW_ID` and the four flags (`AWAITING_INSPECTION`, `BORO_UNKNOWN`, `ZERO_COORDINATES` and `BBL_INCOMPLETE`).

**Why it matters:** the cleaning removed only the duplicate copies. No restaurant and no inspection was lost.

**Part 3. Intention:** I want to save the prepared dataset and the files that explain my cleaning: the before and after counts, the duplicate exclusion log, the column mapping and the explanation of each flag column. Then I read the dataset back from the file to check that the export is correct (same rows, and the identifiers keep their leading zeros).

**Note for anyone who uses the prepared dataset (for example, to continue the cleaning or to start the analysis):** please load the file with the code below. If you use a plain `pd.read_csv`, pandas turns the identifiers (like `Census Tract`, `ZIPCODE` and `BBL`) back into decimals and loses their leading zeros, and the dates come back as plain text. That would undo part of my cleaning. Excel can also remove the leading zeros if you open the CSV there.

```python
pd.read_csv("data/restaurants_prepared.csv",
            dtype={"ZIPCODE": str, "Community Board": str, "Council District": str,
                   "Census Tract": str, "BIN": str, "BBL": str},
            parse_dates=["INSPECTION DATE", "GRADE DATE", "RECORD DATE"],
            low_memory=False)
```

In [63]:
import os
os.makedirs("handoff", exist_ok=True)

# column mapping: no names were changed, 5 columns are new
new_cols = [c for c in df_work.columns if c not in df.columns]
column_mapping = pd.DataFrame({
    "original_name": list(df.columns) + [""] * len(new_cols),
    "prepared_name": list(df.columns) + new_cols,
    "status": ["name unchanged"] * len(df.columns) + ["new column"] * len(new_cols),
})
column_mapping["prepared_type"] = [str(df_work[c].dtype) for c in column_mapping["prepared_name"]]

# what each new column means
flags = ["AWAITING_INSPECTION", "BORO_UNKNOWN", "ZERO_COORDINATES", "BBL_INCOMPLETE"]
flag_explanations = pd.DataFrame({
    "column": ["SOURCE_ROW_ID"] + flags,
    "meaning": [
        "Row number in the original file (1 = first data row). Use it to trace any row back to the raw data.",
        "True when INSPECTION DATE is 1900-01-01 (restaurant not inspected yet). The row and the date are kept.",
        "True when BORO is 0 (unknown borough). BORO keeps its original value.",
        "True when Latitude and Longitude were both 0. In those rows they were set to empty.",
        "True when BBL was a single digit instead of 10 digits (it only repeated the borough code). In those rows BBL was set to empty.",
    ],
})
flag_explanations["rows_true"] = [None] + [int(df_work[f].sum()) for f in flags]

# save the files
df_work.to_csv("data/restaurants_prepared.csv", index=False)
before_after.to_csv("handoff/before_after_counts.csv")
exclusion_log.to_csv("handoff/duplicate_exclusion_log.csv", index=False)
column_mapping.to_csv("handoff/column_mapping.csv", index=False)
flag_explanations.to_csv("handoff/flag_explanations.csv", index=False)

# read the dataset back and check it
check_file = pd.read_csv("data/restaurants_prepared.csv", dtype={c: str for c in text_cols}, low_memory=False)
print("Saved rows and columns:", check_file.shape, "| same rows as df_work:", len(check_file) == len(df_work))
print("Census Tract lengths in the file:", sorted(check_file["Census Tract"].dropna().str.len().unique()))
print("Files in handoff/:", sorted(os.listdir("handoff")))

Saved rows and columns: (295843, 32) | same rows as df_work: True
Census Tract lengths in the file: [np.int64(6)]
Files in handoff/: ['before_after_counts.csv', 'categorical_missing.csv', 'column_mapping.csv', 'data_dictionary.csv', 'duplicate_exclusion_log.csv', 'flag_explanations.csv', 'violation_validation_summary.csv']


**What I saved:**

1. `data/restaurants_prepared.csv`: the prepared dataset (295,843 rows, 32 columns). When my part is complete, I share it with the team as a zipped file on the GitHub Releases page.
2. In the `handoff/` folder:
   - `before_after_counts.csv`: rows, columns, restaurants and inspections before and after my cleaning.
   - `duplicate_exclusion_log.csv`: the 169 rows I removed, with the `SOURCE_ROW_ID` of the row I kept.
   - `column_mapping.csv`: the 27 original columns (no names changed), the 5 new columns and the type of each one.
   - `flag_explanations.csv`: what each new column means and how many rows it marks.

**Check:** I read the dataset back from the file. It has the same 295,843 rows, and the `Census Tract` values all have 6 characters, so the leading zeros are kept.

---

# End of Tasks 1 to 11: Data Preparation and Structural Cleaning

**The next part of the project starts below this line.** Please start from my prepared dataset (`data/restaurants_prepared.csv`) and not from the raw file. Load it with this code, so the identifiers keep their leading zeros and the dates are real dates:

```python
import pandas as pd

df_prepared = pd.read_csv("data/restaurants_prepared.csv",
                          dtype={"ZIPCODE": str, "Community Board": str, "Council District": str,
                                 "Census Tract": str, "BIN": str, "BBL": str},
                          parse_dates=["INSPECTION DATE", "GRADE DATE", "RECORD DATE"],
                          low_memory=False)
```

The files in the `handoff/` folder explain what I changed: `flag_explanations.csv` explains each new column, and `duplicate_exclusion_log.csv` lists the 169 rows I removed.

---